We start with Imports and Setup for PyTorch, torchvision, and some utilities.

In [7]:
# === Cell 1: Imports ===
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit

from pathlib import Path
import os


We set up paths, hyperparameters, and constants.

In [8]:
# === Cell 2: Configs ===

# Root of your repo is the parent of the notebook folder
REPO = Path.cwd().parent   # "690F/"

DATA_DIR = REPO / "data" / "synthetic"
CAPTIONS_CSV = DATA_DIR / "captions.csv"
SPLITS_DIR = REPO / "data" / "splits"
ATTACK_DIR = REPO / "data" / "attack_sets"
OUTPUTS_DIR = REPO / "outputs"

# Classes
CLASSES = ["dogs","cats","food","drink","room","book","painting","outside","car","flowers"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}

# Hyperparameters
BATCH_SIZE = 16
LR = 1e-3
EPOCHS = 5
SEED = 1337

torch.manual_seed(SEED)
np.random.seed(SEED)

print("Repo root:", REPO)
print("Captions CSV:", CAPTIONS_CSV)
print("Splits dir:", SPLITS_DIR)
print("Attack dir:", ATTACK_DIR)
print("Outputs dir:", OUTPUTS_DIR)


Repo root: c:\Users\Rishav\Documents\GitHub\690F
Captions CSV: c:\Users\Rishav\Documents\GitHub\690F\data\synthetic\captions.csv
Splits dir: c:\Users\Rishav\Documents\GitHub\690F\data\splits
Attack dir: c:\Users\Rishav\Documents\GitHub\690F\data\attack_sets
Outputs dir: c:\Users\Rishav\Documents\GitHub\690F\outputs


We’ll make a PyTorch Dataset that reads captions.csv, loads images from disk, applies transforms (resize → tensor → normalize) and finally returns (image, label_index, filename). We need this to connect raw images & labels to the training loop.

In [9]:
# === Cell 3: Validate captions -> files exist ===

df = pd.read_csv(CAPTIONS_CSV)

missing = []
for i, row in df.iterrows():
    fp = REPO / row["filename"]  # expects relative paths like data/synthetic/images/dogs/dog01.jpg
    if not fp.exists():
        missing.append((i, row["filename"]))

print(f"Rows in captions.csv: {len(df)}")
print(f"Missing files: {len(missing)}")

if missing:
    print("⚠️ Some paths in captions.csv don’t match real files. First few:")
    for i, (idx, fn) in enumerate(missing[:5]):
        print(f"  row {idx}: {fn}")
else:
    print("✅ All files in captions.csv exist on disk")


Rows in captions.csv: 0
Missing files: 0
✅ All files in captions.csv exist on disk


We’ll resize images → tensors → normalize to ImageNet stats since pretrained CNNs expect ImageNet-like inputs and normalization helps with stable training.

In [12]:
# === Cell 4: Create splits & attack sets ===

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
ATTACK_DIR.mkdir(parents=True, exist_ok=True)

# check labels are valid
bad_labels = set(df["class"]) - set(CLASSES)
assert not bad_labels, f"Found labels not in CLASSES: {bad_labels}"

X = df["filename"].tolist()
y = df["class"].tolist()

def write_list(path, items):
    with open(path, "w") as f:
        for it in items:
            f.write(str(it) + "\n")

# 70% train, 30% temp
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss.split(X, y))
X_train = [X[i] for i in train_idx]
y_train = [y[i] for i in train_idx]
X_temp  = [X[i] for i in temp_idx]
y_temp  = [y[i] for i in temp_idx]

# split temp equally into val/test (15%/15%)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(sss2.split(X_temp, y_temp))
X_val = [X_temp[i] for i in val_idx]
X_test = [X_temp[i] for i in test_idx]

# save splits
write_list(SPLITS_DIR / "train.txt", X_train)
write_list(SPLITS_DIR / "val.txt",   X_val)
write_list(SPLITS_DIR / "test.txt",  X_test)

# For MIA
write_list(ATTACK_DIR / "member.txt",    X_train)
write_list(ATTACK_DIR / "nonmember.txt", X_test)

print("✅ Wrote splits to:", SPLITS_DIR)
print("Sizes — train/val/test:", len(X_train), len(X_val), len(X_test))
print("✅ Wrote attack sets to:", ATTACK_DIR)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

We’ll use ResNet18 pretrained on ImageNet and fine-tune the last layer for 10 classes.

In [ ]:
# === Cell 5: Dataset Class ===

class MediaTaggerDataset(Dataset):
    def __init__(self, captions_csv, split_file, transform=None):
        self.data = pd.read_csv(captions_csv)
        with open(split_file) as f:
            keep = set([x.strip() for x in f.readlines()])
        self.data = self.data[self.data['filename'].isin(keep)].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = REPO / row['filename']
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = CLASS_TO_IDX[row['class']]
        return image, label, row['filename']


We’ll write a simple training loop with validation to help track overfitting (important for MIA risk).

In [ ]:
# === Cell 6: Transforms & Loaders ===

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

train_set = MediaTaggerDataset(CAPTIONS_CSV, SPLITS_DIR/"train.txt", transform=transform)
val_set   = MediaTaggerDataset(CAPTIONS_CSV, SPLITS_DIR/"val.txt", transform=transform)
test_set  = MediaTaggerDataset(CAPTIONS_CSV, SPLITS_DIR/"test.txt", transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print("Train size:", len(train_set))
print("Val size:", len(val_set))
print("Test size:", len(test_set))


In [ ]:
# === Cell 7: Baseline Model ===

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights="IMAGENET1K_V1")
num_feats = model.fc.in_features
model.fc = nn.Linear(num_feats, len(CLASSES))  # replace classifier head
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Using device:", device)


In [ ]:
# === Cell 8: Training Loop ===

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0,0,0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()*imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds==labels).sum().item()
        total += labels.size(0)
    return total_loss/total, correct/total

def eval_model(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0,0,0
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()*imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds==labels).sum().item()
            total += labels.size(0)
    return total_loss/total, correct/total

best_val_acc = 0
for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = eval_model(model, val_loader, criterion)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train acc {tr_acc:.3f} | Val acc {val_acc:.3f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        (OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), OUTPUTS_DIR/"models/classifier_best.pt")
